# Milestone 4 — Multiple-Choice Classification, LoRA & Fine-Tuning



## Setup

In [1]:
!pip install -U torchao --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
options = ["A", "B", "C", "D", "E"]

## Q1 — Label Encoding

Map the `answer` column to numeric labels: A=0, B=1, C=2, D=3, E=4.

In [3]:
label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
train["label"] = train["answer"].map(label_map)

print("Encoded label for row 150:", train.iloc[150]["label"])

Encoded label for row 150: 2


## Q2 — Prompt-Option Formatting

For row 0, build the Option B input as `prompt + " [SEP] " + option_B`.

In [4]:
row0 = train.iloc[0]
option_b_input = str(row0["prompt"]) + " [SEP] " + str(row0["B"])

print(option_b_input)
print("Character length:", len(option_b_input))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Character length: 407


## Q3 — Single-Row MCQ Tokenization

Tokenize all 5 formatted options for row 0, then reshape to `[1, 5, 128]` (batch × choices × sequence length).

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

formatted_inputs = [str(row0["prompt"]) + " [SEP] " + str(row0[opt]) for opt in options]

encoded = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoded["input_ids"].unsqueeze(0)  # add batch dimension -> [1, 5, 128]
print("input_ids shape:", input_ids.shape)
print("Second dimension:", input_ids.shape[1])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

input_ids shape: torch.Size([1, 5, 128])
Second dimension: 5


## Q4 — Batch MCQ Tokenization

Tokenize the first 16 rows (5 choices each) -> shape `[16, 5, 128]`. Count total token positions.

In [6]:
def format_row(row):
    return [str(row["prompt"]) + " [SEP] " + str(row[opt]) for opt in options]

all_formatted = []
for _, row in train.iloc[:16].iterrows():
    all_formatted.extend(format_row(row))

encoded_16 = tokenizer(
    all_formatted,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids_16 = encoded_16["input_ids"].view(16, 5, 128)
print("input_ids shape:", input_ids_16.shape)
print("Total token positions:", input_ids_16.numel())

input_ids shape: torch.Size([16, 5, 128])
Total token positions: 10240


## Q5 — Multiple-Choice Logits

`AutoModelForMultipleChoice` outputs one logit per option -> shape `[1, 5]`.

In [7]:
from transformers import AutoModelForMultipleChoice

mc_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
mc_model.eval()

mc_input_ids = encoded["input_ids"].unsqueeze(0)
mc_attention_mask = encoded["attention_mask"].unsqueeze(0)

with torch.no_grad():
    outputs = mc_model(input_ids=mc_input_ids, attention_mask=mc_attention_mask)

print("Logits shape:", outputs.logits.shape)
print("Number of logits:", outputs.logits.shape[1])

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 5])
Number of logits: 5


## Q6 — Supervised Loss Tensor

Pass the correct label along with the input. The model returns a scalar loss (0-dimensional tensor).

In [8]:
label_0 = torch.tensor([row0["label"]])

with torch.no_grad():
    outputs_with_loss = mc_model(
        input_ids=mc_input_ids,
        attention_mask=mc_attention_mask,
        labels=label_0
    )

print("Loss value:", outputs_with_loss.loss.item())
print("Loss tensor dimensions:", outputs_with_loss.loss.dim())

Loss value: 1.606817603111267
Loss tensor dimensions: 0


## Q7 — LoRA Trainable Parameters

LoRA freezes the base model and only trains small adapter matrices, drastically cutting the number of trainable parameters.

In [9]:
pip install peft

Note: you may need to restart the kernel to use updated packages.


In [10]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(mc_model, lora_config)

trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print("Trainable parameters:", trainable_params)

Trainable parameters: 295681


## Q8 — Hugging Face Dataset Preparation

Build a `Dataset` from the first 100 rows: each item has `input_ids` and `attention_mask` of shape `[5, 128]`, plus a `labels` field.

In [11]:
from datasets import Dataset

def tokenize_row(row, max_length=128):
    formatted = [str(row["prompt"]) + " [SEP] " + str(row[opt]) for opt in options]
    enc = tokenizer(
        formatted,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return {
        "input_ids": enc["input_ids"].tolist(),
        "attention_mask": enc["attention_mask"].tolist(),
        "labels": row["label"]
    }

records_100 = [tokenize_row(row) for _, row in train.iloc[:100].iterrows()]
hf_dataset = Dataset.from_list(records_100)

first_item_ids = hf_dataset[0]["input_ids"]
print("input_ids shape:", (len(first_item_ids), len(first_item_ids[0])))
print("Number of tokenized choices:", len(first_item_ids))

input_ids shape: (5, 128)
Number of tokenized choices: 5


## Q9 — Tiny LoRA Fine-Tuning

Fine-tune the LoRA multiple-choice model on the first 32 rows with `max_length=64`, batch size 4, and only 4 training steps.

In [12]:
from transformers import TrainingArguments, Trainer
from dataclasses import dataclass
from typing import Any, Dict, List

# Custom collator: multiple-choice batches need input_ids/attention_mask stacked as (batch, choices, seq_len)
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: Any
    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        labels = [f.pop("labels") for f in features]
        return {
            "input_ids": torch.tensor([f["input_ids"] for f in features]),
            "attention_mask": torch.tensor([f["attention_mask"] for f in features]),
            "labels": torch.tensor(labels)
        }

data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

train_records_32 = [tokenize_row(row, max_length=64) for _, row in train.iloc[:32].iterrows()]
train_dataset_32 = Dataset.from_list(train_records_32)

# Fresh base model + LoRA adapter for training
mc_model_ft = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
lora_model_ft = get_peft_model(mc_model_ft, lora_config)

training_args = TrainingArguments(
    output_dir="./mc_lora_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none"
)

trainer = Trainer(
    model=lora_model_ft,
    args=training_args,
    train_dataset=train_dataset_32,
    data_collator=data_collator
)

trainer.train()
print("Final global_step:", trainer.state.global_step)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,3.234258
2,3.383560
3,3.173378
4,3.311106


Final global_step: 4


## Q10 — Probability Assigned to Option E After Fine-Tuning

Run the fine-tuned model on row 0 and apply softmax to get option probabilities.

In [13]:
import torch.nn.functional as F

row0_64 = tokenize_row(row0, max_length=64)
infer_input_ids = torch.tensor([row0_64["input_ids"]])
infer_attention_mask = torch.tensor([row0_64["attention_mask"]])

# Move inputs to whatever device the fine-tuned model is currently on (CPU or GPU)
device = next(lora_model_ft.parameters()).device
infer_input_ids = infer_input_ids.to(device)
infer_attention_mask = infer_attention_mask.to(device)

lora_model_ft.eval()
with torch.no_grad():
    infer_outputs = lora_model_ft(input_ids=infer_input_ids, attention_mask=infer_attention_mask)

probs = F.softmax(infer_outputs.logits, dim=1)
print("Probabilities (A-E):", probs)

option_e_prob = probs[0][4].item()  # E is index 4
print("Probability of Option E:", round(option_e_prob, 4))

Probabilities (A-E): tensor([[0.1949, 0.2034, 0.2004, 0.2022, 0.1991]], device='cuda:0')
Probability of Option E: 0.1991
